In [ ]:
%pip install  "langchain[openai]" "langchain[mistralai]" "langchain-community" 


Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
%pip install faiss-cpu

   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.1/18.9 MB 975.2 kB/s eta 0:00:20
    --------------------------------------- 0.4/18.9 MB 3.4 MB/s eta 0:00:06
   - -------------------------------------- 0.7/18.9 MB 4.4 MB/s eta 0:00:05
   -- ------------------------------------- 1.1/18.9 MB 5.4 MB/s eta 0:00:04
   --- ------------------------------------ 1.6/18.9 MB 6.4 MB/s eta 0:00:03
   ---- ----------------------------------- 2.1/18.9 MB 7.0 MB/s eta 0:00:03
   ----- ---------------------------------- 2.6/18.9 MB 7.5 MB/s eta 0:00:03
   ------- -------------------------------- 3.4/18.9 MB 8.6 MB/s eta 0:00:02
   -------- ------------------------------- 4.2/18.9 MB 9.5 MB/s eta 0:00:02
   ---------- ----------------------------- 5.2/18.9 MB 10.6 MB/s eta 0:00:02
   ------------- -------------------------- 6.3/18.9 MB 11.9 MB/s eta 0:00:02
   ------


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import requests
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="mistralai/ministral-3-3b",
    model_provider="mistralai",
    base_url ="http://localhost:1234/v1",
    api_key = "not needed"
)
response = model.invoke('what is the capital of france?')
print(response.content)



content='The capital of France is **Paris**.' additional_kwargs={} response_metadata={'token_usage': {'prompt_tokens': 542, 'completion_tokens': 9, 'total_tokens': 551}, 'model_name': 'mistralai/ministral-3-3b', 'model': 'mistralai/ministral-3-3b', 'finish_reason': 'stop', 'model_provider': 'mistralai'} id='lc_run--019c9480-ba47-7453-8516-3e1eb9e90d79-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 542, 'output_tokens': 9, 'total_tokens': 551}


In [30]:
# PydanticOutputParser — gives you Pydantic objects with validation

# JsonOutputParser — parses JSON object strings into Python dict

# StructuredOutputParser + schemas — generate structured formats guided by

from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser, JsonOutputParser , SimpleJsonOutputParser
from langchain_core.prompts import PromptTemplate
class AnswerSchema(BaseModel):
    answer: str = Field(..., description="The answer to the question")

structured_data  = model.with_structured_output(AnswerSchema)
#2 create pydantic parser
parser = PydanticOutputParser(pydantic_object=AnswerSchema)
instructions = parser.get_format_instructions()

#3 create a prompt template 

prompt  = PromptTemplate(
    template = "Answer the following question:\n{question}\n{format_instructions}",
    input_variables=["question"],
    partial_variables={"format_instructions": instructions},

)

#4 chain prompt

chain = prompt | model | parser
result = chain.invoke({"question": "What is trigeminal neuralgia?"})
# result = structured_data.invoke("What is trigeminal neuralgia?") #model doesnt support this
print(result)

answer='\nTrigeminal neuralgia (TN) is a medical condition characterized by severe, sudden stabbing or electric-like pain along one or more branches of the trigeminal nerve (cranial nerve V). This pain often occurs in brief episodes and can be triggered by simple activities such as brushing teeth, shaving, exposed to wind, smiling, or even accidental touching of the face.\n\nKey features include:\n- **Location**: Pain typically affects the cheek, forehead, or eye area but does not always follow a predictable pattern.\n- **Triggers**: Common triggers include cold air/water exposure, chewing, laughing, or facial movements.\n- **Duration**: Attacks last only seconds to minutes and can be very painful.\n\nCauses of trigeminal neuralgia are often unknown, but it is commonly associated with:\n- Compression by a blood vessel on the nerve root\n- Multiple sclerosis (MS) or other demyelinating diseases\n- Tumors affecting the brainstem or cerebellum\n\nDiagnosis involves ruling out other condit

In [ ]:
# import requests

# response = requests.post(
#     "http://localhost:1234/v1/embeddings",
#     json={
#         "model": "text-embedding-embeddinggemma-300m-qat",
#         "input": "test"
#     }
# )

# print(response.json())

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",  # can be anything
    model="text-embedding-embeddinggemma-300m-qat", # LM Studio ignores name but keep something
    check_embedding_ctx_length=False  
)

vector = embeddings.embed_documents(["test","test2"])
print(len(vector))



2


In [42]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_core.tools import create_retriever_tool
from pprint import pprint
docs = [
    Document(
        page_content=(
            "LangChain is a framework for building applications powered by large language models. "
            "It provides abstractions for prompts, models, memory, and retrieval."
        ),
        metadata={
            "source": "https://python.langchain.com/docs/introduction/",
            "title": "LangChain Documentation - Introduction",
            "doc_type": "docs",
            "retrieved_at": "2026-02-25"
        },
    ),
    Document(
        page_content=(
            "RAG combines retrieval and generation by fetching relevant context from a knowledge base "
            "before producing an answer."
        ),
        metadata={
            "source": "https://arxiv.org/abs/2005.11401",
            "title": "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
            "doc_type": "paper",
            "retrieved_at": "2026-02-25"
        },
    ),
    Document(
        page_content=(
            "Vector databases store embeddings and support similarity search, "
            "which is useful for semantic retrieval in RAG pipelines."
        ),
        metadata={
            "source": "internal://rag-handbook/chapter-2",
            "title": "RAG Handbook - Chapter 2",
            "doc_type": "internal_note",
            "retrieved_at": "2026-02-25"
        },
    ),
]
text =[
    'I love apple',
    'sometimes talking is good',
    'apple phone has great ecosystem for mobile users',
    'I think pear tastes better than apple',
    'pear"s soap has a nice smell',
    'November is the best month of the year'
]
vec_text = FAISS.from_texts(text,embeddings)
vectorstore = FAISS.from_documents(docs,embeddings)
results = vectorstore.similarity_search(query="What is RAG?", k=4, fetch_k= 3)
res_text = vec_text.similarity_search(query="I like apple but not pear", k=3, fetch_k=2)
retreiver = vectorstore.as_retriever( search_type="mmr",search_kwargs={"k": 4, "fetch_k": 3})
retreiver_tool = create_retriever_tool(retriever=retreiver, name="RAG_Retreiver", description="Use this tool to retrieve relevant documents about RAG")
# pprint(retreiver_tool.invoke("What is RAG?"))
pprint(res_text)



[Document(id='32ae5d78-ef37-41bc-87a6-fa83011f072d', metadata={}, page_content='I think pear tastes better than apple'),
 Document(id='311265a4-f3f5-49d1-bb80-0f6e9e983428', metadata={}, page_content='I love apple'),
 Document(id='a22bc850-d0d3-4544-99f1-5baac39f6384', metadata={}, page_content='pear"s soap has a nice smell')]


In [ ]:
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

index_to_docstore_id = {}

dimension = len(embeddings.embed_query("what is RAG"))
index = faiss.IndexHNSWFlat(dimension, 32)  # Using HNSW index for efficient similarity search
index.hnsw.efConstruction = 200  # Higher values improve recall at the cost of indexing time
index.hnsw.efSearch = 50  #How deep we are going through the layers while searching the graph Higher values improve recall at the cost of search time
vector_store =FAISS(embedding_function=embeddings, index=index, docstore=InMemoryDocstore({}), index_to_docstore_id=index_to_docstore_id)

vector_store.add_documents(docs)
result= vector_store.similarity_search(query="What is RAG?", k=4, fetch_k=3)
pprint(result)



[Document(id='fe762f32-44a5-48f3-9ac2-c51b84bc54d9', metadata={'source': 'https://arxiv.org/abs/2005.11401', 'title': 'Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks', 'doc_type': 'paper', 'retrieved_at': '2026-02-25'}, page_content='RAG combines retrieval and generation by fetching relevant context from a knowledge base before producing an answer.'),
 Document(id='926e9618-7428-4754-b768-03a904f29403', metadata={'source': 'internal://rag-handbook/chapter-2', 'title': 'RAG Handbook - Chapter 2', 'doc_type': 'internal_note', 'retrieved_at': '2026-02-25'}, page_content='Vector databases store embeddings and support similarity search, which is useful for semantic retrieval in RAG pipelines.'),
 Document(id='0c3dc5d6-5260-483c-8233-ffeff12e8f15', metadata={'source': 'https://python.langchain.com/docs/introduction/', 'title': 'LangChain Documentation - Introduction', 'doc_type': 'docs', 'retrieved_at': '2026-02-25'}, page_content='LangChain is a framework for building appl

In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRequest, ModelResponse , wrap_model_call
from dotenv import load_dotenv
from langchain.messages import SystemMessage, HumanMessage

load_dotenv()

basic_model = model

@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:

    message_count  = len(request.state['messages'])
    if message_count > 3:
        selected_model = basic_model
    else:
        selected_model = basic_model

    request.override(model=selected_model)

    return handler(request)

agent  = create_agent(model = basic_model
                      ,middleware = [dynamic_model_selection])

response = agent.invoke({"messages": [SystemMessage(content="You are a helpful assistant."),
                        HumanMessage(content="who is the cruel dictator ever in history?")]})

print(response['messages'][-1].content)

c:\Users\stony\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'model' is not defined